# ANN Playground — Perceptron Multicouches et décrochage étudiant

## 1. Introduction

## 2. Veille sur le MLP

## 3. TensorFlow Playground

### Protocole commun

Pour pouvoir comparer, tous les essais partent des mêmes réglages : classification, 50 % de données d'entraînement, bruit 0, batch 10, learning rate 0,03, activation Tanh, pas de régularisation. La graine du Playground est fixée (`seed=0.46416`), donc les points et les poids de départ sont identiques d'un essai à l'autre. Un seul paramètre change à la fois. Chaque essai s'arrête à **500 époques** (1 000 pour la spirale), et je relève la *training loss* et la *test loss* affichées par le site (erreur quadratique moyenne : 0 = parfait, environ 0,5 = réponse au hasard).

Toutes les captures sont dans `images/`, décrites une par une dans `images/README.md` avec le lien qui rejoue la configuration exacte.

### 3.1 Point 1 : nombre de neurones (1 couche cachée, entrées X1 et X2)

Test loss à 500 époques :

| Dataset | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Circle | 0,421 | 0,273 | 0,013 | 0,022 | 0,014 | 0,012 | 0,010 | 0,010 |
| Exclusive or | 0,398 | 0,170 | 0,059 | 0,014 | 0,016 | 0,010 | 0,008 | 0,009 |
| Gaussian | 0,002 | 0,003 | 0,003 | 0,003 | 0,003 | 0,003 | 0,003 | 0,003 |

**Circle** : 1, 3 puis 8 neurones

![Circle, X1 et X2, 1 couche de 1 neurone](images/j3_p1_circle_1neurone.png)
![Circle, X1 et X2, 1 couche de 3 neurones](images/j3_p1_circle_3neurones.png)
![Circle, X1 et X2, 1 couche de 8 neurones](images/j3_p1_circle_8neurones.png)

**Exclusive or** : 1, 3 puis 8 neurones

![XOR, X1 et X2, 1 couche de 1 neurone](images/j3_p1_xor_1neurone.png)
![XOR, X1 et X2, 1 couche de 3 neurones](images/j3_p1_xor_3neurones.png)
![XOR, X1 et X2, 1 couche de 8 neurones](images/j3_p1_xor_8neurones.png)

**Gaussian** : 1, 3 puis 8 neurones

![Gaussian, X1 et X2, 1 couche de 1 neurone](images/j3_p1_gaussian_1neurone.png)
![Gaussian, X1 et X2, 1 couche de 3 neurones](images/j3_p1_gaussian_3neurones.png)
![Gaussian, X1 et X2, 1 couche de 8 neurones](images/j3_p1_gaussian_8neurones.png)

**Interprétation**

- **Gaussian** : un seul neurone suffit (test loss 0,002). Un neurone trace une droite, et les deux nuages sont séparables par une droite. Ajouter des neurones ne change rien : la capacité supplémentaire est inutile pour un problème linéairement séparable.
- **Circle** : 1 neurone (0,421) et 2 neurones (0,273) échouent, puis le saut se produit à **3 neurones** (0,013). Il faut au moins trois droites pour fermer un triangle autour du disque central ; on les voit dans les trois vignettes de la couche cachée. Au-delà, le gain est faible (0,010 avec 8) : la frontière devient plus ronde, mais le problème est déjà résolu.
- **Exclusive or** : l'amélioration est plus progressive : 0,398 → 0,170 → 0,059 → **0,014 à 4 neurones**. Deux droites en croix suffiraient en théorie, mais l'entraînement trouve plus facilement une bonne solution avec un peu plus de neurones.
- **À retenir** : le nombre de neurones nécessaire dépend de la complexité de la frontière de décision. Trop peu de neurones donne du sous-apprentissage ; trop de neurones n'apporte rien sur des données propres (et augmenterait le risque de sur-apprentissage sur des données bruitées).

### 3.2 Point 2 : changer les features d'entrée (0 couche cachée)

On enlève la couche cachée : le réseau ne peut alors tracer qu'une frontière « droite » dans l'espace des features. On cherche la plus petite combinaison de features qui résout chaque dataset.

| Dataset | Features | Train loss | Test loss |
| --- | --- | --- | --- |
| Circle | X1, X2 | 0,499 | 0,500 |
| Circle | X1X2 | 0,499 | 0,509 |
| Circle | **X1², X2²** | 0,001 | 0,001 |
| Circle | les 7 | 0,001 | 0,002 |
| Exclusive or | X1, X2 | 0,455 | 0,584 |
| Exclusive or | X1², X2² | 0,806 | 0,907 |
| Exclusive or | **X1X2** | 0,004 | 0,001 |
| Exclusive or | les 7 | 0,004 | 0,002 |
| Gaussian | **X1, X2** | 0,000 | 0,003 |
| Gaussian | X1², X2² | 0,639 | 0,611 |

En gras : la plus petite combinaison qui résout le dataset.

**Les bonnes features, sans couche cachée**

![Circle, X1² et X2², 0 couche cachée](images/j3_p2_circle_X1carre_X2carre_0couche.png)
![XOR, X1X2 seul, 0 couche cachée](images/j3_p2_xor_X1X2_0couche.png)
![Gaussian, X1 et X2, 0 couche cachée](images/j3_p2_gaussian_X1_X2_0couche.png)

**Contre-exemples : une feature qui détruit l'information**

![Gaussian, X1² et X2², 0 couche cachée](images/j3_p2_gaussian_X1carre_X2carre_0couche.png)
![XOR, X1² et X2², 0 couche cachée](images/j3_p2_xor_X1carre_X2carre_0couche.png)
![Circle, X1 et X2, 0 couche cachée](images/j3_p2_circle_X1_X2_0couche.png)

**Interprétation**

- **Circle** : X1² et X2² suffisent, sans aucun neurone caché (test loss 0,001). X1² + X2² est le carré de la distance au centre. Dans l'espace (X1², X2²), le disque devient un coin et une droite le sépare. C'est exactement ce que faisaient les 3 neurones du point 1 : la bonne feature les remplace.
- **Exclusive or** : X1X2 seul suffit (0,001), car X1X2 est positif quand X1 et X2 ont le même signe, ce qui est la définition même de la classe.
- **Gaussian** : X1 et X2 suffisent déjà. À l'inverse, X1² et X2² le rendent impossible (0,611) : le point (2, 2) et le point (−2, −2) deviennent identiques une fois mis au carré, alors qu'ils sont de classes opposées. Une transformation qui efface l'information utile ne peut pas être rattrapée par le réseau.
- **Toutes les features** marchent aussi, mais on ne sait plus laquelle fait le travail. L'épaisseur des traits sur les captures montre que le réseau a surtout utilisé X1²/X2² (Circle) et X1X2 (XOR).
- **À retenir** : le feature engineering injecte de la connaissance du problème et peut remplacer des couches entières. Pour le projet de décrochage, cela correspond à construire des variables pertinentes (par exemple un taux de réussite par semestre) plutôt que d'agrandir le réseau.

### 3.3 Point 3 : fonction d'activation (Circle, 1 couche de 4 neurones)

| Activation | Train loss | Test loss | Forme de la frontière |
| --- | --- | --- | --- |
| Tanh | 0,008 | 0,022 | Frontière arrondie et lisse. |
| ReLU | 0,001 | 0,006 | Frontière en polygone (segments droits), meilleure test loss ici. |
| Sigmoid | 0,015 | 0,026 | Y arrive aussi en 500 époques, mais plus lentement au début. |
| Linear | 0,499 | 0,500 | Échec total : sans non-linéarité, le réseau entier reste une droite. |

<table><tr><th>Tanh</th><th>ReLU</th></tr><tr><td><img src="images/j4_p3_circle_4neurones_tanh.png" width="480"></td><td><img src="images/j4_p3_circle_4neurones_relu.png" width="480"></td></tr></table>

<table><tr><th>Sigmoid</th><th>Linear</th></tr><tr><td><img src="images/j4_p3_circle_4neurones_sigmoid.png" width="480"></td><td><img src="images/j4_p3_circle_4neurones_linear.png" width="480"></td></tr></table>

**Interprétation**

- **ReLU** obtient la meilleure test loss (0,006). Sa dérivée vaut 1 pour les entrées positives, donc le gradient passe sans être atténué. Sa frontière est faite de segments droits (un polygone), car ReLU est linéaire par morceaux.
- **Tanh** (0,022) donne une frontière arrondie et lisse ; elle est centrée sur 0, comme les données.
- **Sigmoid** (0,026) y arrive aussi avec une seule couche, mais plus lentement. Sa dérivée ne dépasse pas 0,25, ce qui ralentit l'apprentissage ; c'est ce qui posera problème dans un réseau profond (vanishing gradient, point 5).
- **Linear** échoue (0,500, soit le hasard) : une composition de fonctions linéaires reste linéaire. Sans activation non linéaire, un réseau, même profond, ne trace qu'une droite.

### 3.4 Point 4 : learning rate (Circle, 4 neurones, Tanh)

| Learning rate | Test loss : début → 100 → 200 → 500 époques | Comportement |
| --- | --- | --- |
| 0,00001 | 0,537 → 0,536 → 0,535 → 0,532 | Ne bouge pas |
| 0,001 | 0,534 → 0,477 → 0,427 → 0,283 | Descend lentement |
| 0,03 | 0,485 → 0,041 → 0,028 → 0,022 | Descend bien |
| 0,3 | 0,247 → 0,016 → 0,010 → 0,009 | Descend vite et bien |
| 3 | 0,537 → 0,207 → 0,010 → 0,009 | Zigzag (la loss remonte de 0,152 à 0,185 entre 60 et 120 époques) puis se stabilise |
| 10 | 0,898 → 0,870 → 0,840 → 0,800 | Explose puis se bloque |

<table><tr><th>0,00001 : trop petit</th><th>0,03 : bon</th></tr><tr><td><img src="images/j4_p4_circle_lr_0_00001.png" width="480"></td><td><img src="images/j4_p4_circle_lr_0_03.png" width="480"></td></tr></table>

<table><tr><th>3 : zigzag</th><th>10 : explose</th></tr><tr><td><img src="images/j4_p4_circle_lr_3.png" width="480"></td><td><img src="images/j4_p4_circle_lr_10.png" width="480"></td></tr></table>

![Circle, 4 neurones, Tanh, learning rate 0,001](images/j4_p4_circle_lr_0_001.png)
![Circle, 4 neurones, Tanh, learning rate 0,3](images/j4_p4_circle_lr_0_3.png)

**Interprétation**

À chaque mise à jour, chaque poids est corrigé de w ← w − η × gradient, où η est le learning rate.

- **Trop petit (0,00001 ; 0,001)** : la direction est bonne mais les pas sont minuscules. La loss baisse à peine en 500 époques ; il faudrait des dizaines de milliers d'époques.
- **Bon (0,03 ; 0,3)** : descente régulière. Sur ce problème simple, 0,3 est même plus rapide et finit meilleur.
- **Trop grand (3)** : les pas sautent par-dessus le fond de la vallée. La loss remonte par moments (zigzag) avant de se stabiliser par chance.
- **Beaucoup trop grand (10)** : les poids deviennent énormes, les neurones Tanh saturent à ±1, leur gradient devient nul et le réseau se bloque, pire que le hasard.
- **À retenir** : il existe une plage utile, qui dépend du problème. Le learning rate se règle par essais, en regardant la courbe de loss (en Keras : optimiseur Adam, `EarlyStopping`).

### 3.5 Point 5 : la spirale (1 000 époques)

| Configuration | Train loss | Test loss | Remarque |
| --- | --- | --- | --- |
| X1 et X2, 1 couche de 8, Tanh | 0,228 | 0,313 | Une couche ne suffit pas pour une spirale. |
| X1 et X2, 2 couches de 8, Tanh | 0,034 | 0,191 | L'entraînement est bon mais le test non : le réseau commence à sur-apprendre. |
| les 7 features, 2 couches de 8, Tanh | 0,001 | 0,006 | Meilleure configuration mesurée. |
| X1, X2, sin(X1), sin(X2), 3 couches de 8, Tanh | 0,000 | 0,020 | Très bon : les sinus apportent la périodicité des bras. |
| X1 et X2, 4 couches de 8, ReLU | 0,000 | 0,028 | Avec assez de profondeur et ReLU, X1 et X2 suffisent ; frontière en escalier typique de ReLU. |
| X1, X2, sin, 2 couches de 6, ReLU, lr 0,01 | 0,016 | 0,037 | Plus petit réseau et pas plus petits : fonctionne mais plus lentement. |

![Spirale, X1 et X2, 1 couche de 8, Tanh](images/j4_p5_spirale_1couche8_X1X2_tanh.png)
![Spirale, X1 et X2, 2 couches de 8, Tanh](images/j4_p5_spirale_2couches8_X1X2_tanh.png)
![Spirale, les 7 features, 2 couches de 8, Tanh](images/j4_p5_spirale_2couches8_toutes_tanh.png)
![Spirale, X1, X2, sin(X1), sin(X2), 3 couches de 8, Tanh](images/j4_p5_spirale_3couches8_X1X2sin_tanh.png)
![Spirale, X1 et X2, 4 couches de 8, ReLU](images/j4_p5_spirale_4couches8_X1X2_relu.png)
![Spirale, X1, X2, sin, 2 couches de 6, ReLU, lr 0,01](images/j4_p5_spirale_2couches6_X1X2sin_relu_lr0_01.png)

**Interprétation**

- **Une seule couche ne suffit pas** avec X1 et X2 (0,313) : la spirale demande de combiner des combinaisons de droites, donc de la profondeur.
- **2 couches avec X1, X2** : la training loss devient bonne (0,034) mais pas la test loss (0,191). Le réseau apprend les points d'entraînement sans bien généraliser entre les bras : c'est du sur-apprentissage.
- **Les features changent tout** : avec les 7 features, 2 couches donnent 0,006 ; avec sin(X1) et sin(X2), 3 couches donnent 0,020. Les sinus apportent directement la périodicité des bras de la spirale.
- **ReLU avec de la profondeur** (4 couches) y arrive avec X1 et X2 seuls (0,028), avec une frontière en escalier caractéristique.
- **Meilleure configuration mesurée** : les 7 features, 2 couches de 8 neurones, Tanh (test loss 0,006).

### 3.6 Le vanishing gradient

Même réseau dans les trois cas : spirale, X1 et X2, **6 couches cachées de 8 neurones**, learning rate 0,03, 1 000 époques. Seule l'activation change.

| Activation | Test loss : début → 500 → 1 000 époques |
| --- | --- |
| Sigmoid | 0,500 → 0,500 → 0,500 |
| ReLU | 0,499 → 0,033 → 0,026 |
| Tanh | 0,479 → 0,284 → 0,097 |

<table><tr><th>Sigmoid : n'apprend rien</th><th>ReLU : apprend la spirale</th></tr><tr><td><img src="images/j4_vg_spirale_6couches8_sigmoid.png" width="480"></td><td><img src="images/j4_vg_spirale_6couches8_relu.png" width="480"></td></tr></table>

![Spirale, X1 et X2, 6 couches de 8, Tanh, lr 0,03, 1 000 époques](images/j4_vg_spirale_6couches8_tanh.png)

**Interprétation**

Avec **Sigmoid**, la loss reste **exactement à 0,500 pendant 1 000 époques** : le réseau n'apprend rien, la carte de sortie reste grise et uniforme, et les vignettes des premières couches ne se forment jamais. Avec **ReLU**, le même réseau descend à 0,026.

Pourquoi : la rétropropagation calcule le gradient d'une couche en multipliant celui de la couche suivante par la dérivée de l'activation et par les poids. La dérivée de la sigmoïde vaut au maximum 0,25. En remontant 6 couches, le signal d'erreur est multiplié au moins 6 fois par 0,25 (0,25⁶ ≈ 0,0002), et encore par des poids initiaux petits. Il arrive presque nul dans les premières couches : leurs poids ne bougent pas, et comme toutes les couches suivantes dépendent d'elles, le réseau entier stagne. C'est le **vanishing gradient** (disparition du gradient).

- **ReLU** a une dérivée égale à 1 pour les entrées positives : le gradient traverse les couches sans être écrasé.
- **Tanh** (dérivée maximale de 1) est un cas intermédiaire : bloquée 300 à 400 époques sur un plateau, elle finit par apprendre (0,097 à 1 000 époques).
- **Remèdes** : ReLU et ses variantes, une bonne initialisation des poids (He, Glorot), la batch normalization, les connexions résiduelles, ou un réseau moins profond.

### 3.7 Ce que je retiens pour mon propre MLP

- Commencer avec un réseau petit (1 à 2 couches) : la capacité nécessaire dépend de la complexité de la frontière, et plus gros n'est pas mieux.
- Soigner les variables d'entrée : une bonne feature vaut plusieurs neurones, une mauvaise transformation détruit l'information.
- Utiliser ReLU dans les couches cachées pour éviter le vanishing gradient ; garder Sigmoid (ou Softmax) seulement pour la sortie.
- Régler le learning rate en regardant la courbe de loss : trop petit = trop lent, trop grand = zigzag ou explosion.
- Toujours juger sur la loss de validation ou de test, pas sur la loss d'entraînement (cf. spirale à 2 couches).

## 4. Les données

### 4.1 D'où viennent les données

Le jeu **Predict Students' Dropout and Academic Success** est publié sur l'UCI Machine Learning Repository (identifiant **697**, licence CC BY 4.0). Il a été constitué par des chercheurs de l'**Instituto Politécnico de Portalegre** (Portugal), dans un projet financé par le programme SATDAP. L'objectif était de repérer **tôt** les étudiants en risque d'échec ou d'abandon, pour leur proposer un accompagnement.

- **Une ligne = un étudiant** inscrit en licence (agronomie, design, éducation, soins infirmiers, journalisme, gestion, service social, technologies…) : **4 424 étudiants**.
- **36 variables explicatives** : ce que l'on sait à l'inscription (parcours scolaire, données démographiques et socio-économiques), les résultats académiques à la fin du 1er et du 2e semestre, et des indicateurs macro-économiques.
- **La cible `Target`** a 3 classes, à la fin de la durée normale du cursus : **Dropout** (a abandonné), **Enrolled** (encore inscrit, en retard) et **Graduate** (diplômé). C'est donc une **classification à 3 classes**, avec un déséquilibre annoncé par les auteurs.
- Les auteurs indiquent avoir déjà traité les valeurs manquantes et les anomalies, et recommandent un découpage 80 % entraînement / 20 % test.

![Page UCI du jeu de données 697](images/j5_page_uci_dataset_697.png)

*Sources : Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). Predict Students' Dropout and Academic Success [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89 ; Martins M.V. et al. (2021), « Early prediction of student's performance in higher education: a case study », Springer, DOI 10.1007/978-3-030-72657-7_16.*

### 4.2 Chargement

On télécharge les données avec `ucimlrepo`, puis on garde une copie locale dans `data/data.csv` pour ne pas dépendre d'Internet à chaque exécution. Le fichier d'origine de l'UCI utilise le **point-virgule** comme séparateur ; on garde la même convention.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

CSV = "data/data.csv"

if os.path.exists(CSV):
    df = pd.read_csv(CSV, sep=";")
else:
    from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=697)
    df = pd.concat([data.data.features, data.data.targets], axis=1)
    os.makedirs("data", exist_ok=True)
    df.to_csv(CSV, sep=";", index=False)

# Certains noms de colonnes ont des espaces ou une tabulation en trop dans le CSV d'origine
df.columns = df.columns.str.strip()

print(df.shape)

(4424, 37)


**Lecture :** 4 424 lignes (étudiants) et 37 colonnes (36 variables explicatives + la cible `Target`).

In [2]:
df.head()

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital Status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

**Lecture de `info()` :** aucune valeur manquante (4 424 valeurs non nulles dans chaque colonne). Toutes les variables explicatives sont numériques (`int64` ou `float64`) ; seule la cible est du texte (type `str` avec pandas 3, `object` avec les versions plus anciennes). Attention : « numérique » ne veut pas dire « quantité » (voir 4.4). On note aussi une faute de frappe d'origine dans un nom de colonne : `Nacionality` (on la garde telle quelle pour rester fidèle à la source).

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Marital Status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
Application mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
Application order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
Course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
Daytime/evening attendance,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
Previous qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
Previous qualification (grade),4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
Nacionality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
Mother's qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
Father's qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


**Lecture de `describe()` :**

- `Age at enrollment` va de 17 à 70 ans, avec une médiane autour de 20 ans : la plupart des étudiants entrent juste après le lycée, mais une minorité reprend ses études plus tard.
- `Admission grade` et `Previous qualification (grade)` sont sur 200 ; les notes des semestres (`Curricular units … (grade)`) sont sur 20.
- Les notes de semestre ont un minimum à 0 : ce sont souvent des étudiants qui n'ont passé aucune évaluation, et pas forcément une vraie note de 0. On le regardera à l'EDA.
- Les colonnes codées 0/1 (`Gender`, `Debtor`, `Scholarship holder`…) ont une moyenne qui se lit directement comme une proportion : par exemple la moyenne de `Scholarship holder` donne la part de boursiers.

In [5]:
# La cible : 3 classes
counts = df["Target"].value_counts()
pd.DataFrame({"effectif": counts, "part (%)": (counts / len(df) * 100).round(1)})

,effectif,part (%)
Target,,
Graduate,2209,49.9
Dropout,1421,32.1
Enrolled,794,17.9


**La cible** : Graduate ≈ 50 %, Dropout ≈ 32 %, Enrolled ≈ 18 %. Les classes sont **déséquilibrées** : un modèle qui répondrait toujours « Graduate » aurait déjà 50 % d'exactitude. Il faudra donc juger les modèles avec d'autres métriques que l'accuracy seule (rappel par classe, F1, matrice de confusion).

In [6]:
list(df.columns)

['Marital Status',
 'Application mode',
 'Application order',
 'Course',
 'Daytime/evening attendance',
 'Previous qualification',
 'Previous qualification (grade)',
 'Nacionality',
 "Mother's qualification",
 "Father's qualification",
 "Mother's occupation",
 "Father's occupation",
 'Admission grade',
 'Displaced',
 'Educational special needs',
 'Debtor',
 'Tuition fees up to date',
 'Gender',
 'Scholarship holder',
 'Age at enrollment',
 'International',
 'Curricular units 1st sem (credited)',
 'Curricular units 1st sem (enrolled)',
 'Curricular units 1st sem (evaluations)',
 'Curricular units 1st sem (approved)',
 'Curricular units 1st sem (grade)',
 'Curricular units 1st sem (without evaluations)',
 'Curricular units 2nd sem (credited)',
 'Curricular units 2nd sem (enrolled)',
 'Curricular units 2nd sem (evaluations)',
 'Curricular units 2nd sem (approved)',
 'Curricular units 2nd sem (grade)',
 'Curricular units 2nd sem (without evaluations)',
 'Unemployment rate',
 'Inflation rat

### 4.3 Les 4 familles de variables

| Famille | Variables | Nombre |
| --- | --- | --- |
| **Profil et situation de l'étudiant** | `Marital Status`, `Nacionality`, `Gender`, `Age at enrollment`, `International`, `Displaced` (étudie loin de chez lui), `Educational special needs`, `Mother's qualification`, `Father's qualification`, `Mother's occupation`, `Father's occupation`, `Debtor`, `Tuition fees up to date`, `Scholarship holder` | 14 |
| **Parcours d'entrée** | `Application mode`, `Application order` (rang du vœu, 0 = premier choix), `Course`, `Daytime/evening attendance`, `Previous qualification`, `Previous qualification (grade)`, `Admission grade` | 7 |
| **Résultats du 1er et du 2e semestre** | Pour chaque semestre, `Curricular units … (credited)`, `(enrolled)`, `(evaluations)`, `(approved)`, `(grade)`, `(without evaluations)` | 12 |
| **Contexte économique** | `Unemployment rate`, `Inflation rate`, `GDP` (au moment de l'inscription) | 3 |

**Remarque importante pour la modélisation :** les résultats de semestre ne sont connus qu'**après** un ou deux semestres. Un modèle qui les utilise prédit bien, mais plus tard. Un modèle qui ne garde que le profil et le parcours d'entrée prédit dès l'inscription, mais moins bien. C'est un choix à justifier au moment de construire le MLP.

In [7]:
# Combien de valeurs différentes par colonne ? Aide à repérer les catégories déguisées en nombres
df.nunique().sort_values()

Daytime/evening attendance                          2
Displaced                                           2
Debtor                                              2
Educational special needs                           2
International                                       2
Scholarship holder                                  2
Gender                                              2
Tuition fees up to date                             2
Target                                              3
Marital Status                                      6
Application order                                   8
Inflation rate                                      9
Curricular units 2nd sem (without evaluations)     10
Unemployment rate                                  10
GDP                                                10
Curricular units 1st sem (without evaluations)     11
Course                                             17
Previous qualification                             17
Application mode            

### 4.4 Les catégories déguisées en nombres

Une colonne numérique n'est une **quantité** que si les écarts ont un sens (20 ans → 40 ans, c'est 20 ans de plus). Ici, beaucoup de colonnes sont des **codes** : le cursus n° 9 991 n'est pas « plus grand » que le cursus n° 33, de la même façon que le joueur n° 10 n'est pas deux fois le joueur n° 5.

| Type | Colonnes | Traitement prévu au jour 8 |
| --- | --- | --- |
| **Catégorielles nominales (codes)** | `Marital Status`, `Application mode`, `Course`, `Previous qualification`, `Nacionality`, `Mother's qualification`, `Father's qualification`, `Mother's occupation`, `Father's occupation` | Encodage one-hot (une colonne 0/1 par modalité), en regroupant les modalités très rares |
| **Binaires 0/1** | `Daytime/evening attendance`, `Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`, `Gender`, `Scholarship holder`, `International` | Utilisables telles quelles |
| **Ordinale** | `Application order` (0 = premier choix … 9) | Utilisable telle quelle : l'ordre a un sens |
| **Quantitatives** | `Age at enrollment`, `Previous qualification (grade)`, `Admission grade`, les 12 colonnes `Curricular units …`, `Unemployment rate`, `Inflation rate`, `GDP` | Standardisation (moyenne 0, écart-type 1) |

Pourquoi c'est important pour un réseau de neurones : un neurone multiplie chaque entrée par un poids. Si `Course` vaut 33 ou 9 991, le réseau croira que le second cursus est « 300 fois plus » que le premier, ce qui n'a aucun sens. L'encodage one-hot supprime ce faux ordre.

In [8]:
# Exemple : les codes de la colonne Course et leurs effectifs
df["Course"].value_counts()

Course
9500    766
9147    380
9238    355
9085    337
9773    331
9991    268
9670    268
9254    252
9070    226
171     215
8014    215
9003    210
9853    192
9119    170
9130    141
9556     86
33       12
Name: count, dtype: int64

**Bilan du jour 5 :** 4 424 étudiants, 36 variables explicatives (dont 9 catégorielles codées en nombres, 8 binaires, 1 ordinale et 18 quantitatives), aucune valeur manquante, et une cible à 3 classes déséquilibrées (Graduate, Dropout, Enrolled). Prochaine étape : l'analyse exploratoire variable par variable (jour 6).

## 5. Analyse exploratoire

## 6. Préparation

## 7. MLP avec Keras

## 8. Mon MLP en NumPy

## 9. Comparaison

## 10. Conclusion